## ML con Regresión Logística

Rama de la IA que enseña a la maquinas o computdoras a aprender de los datos sin ser programados.


Datos + Respuestas -------------> El modelo aprender las reglas por sí solo


Clasificación ---> Regresión logísticas: Predecir una categorías ( Aprueba o reprueba)

In [1]:
#Liberías e Importaciones

import pandas as pd
import numpy as np

#Visualización
import matplotlib.pyplot as plt
import seaborn as sns

#Modelos de Marchine Learning
from sklearn.linear_model import LinearRegression, LogisticRegression

#train_test_split: Dividir el dataset en datos de entrenamiento y prueba de forma aleatoria y controlada
from sklearn.model_selection import train_test_split

#StarndarScaler
#OrdinalEncoder: Codifique variables categoricas con orden real
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

#Metricas de evaluación para regresión
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#Clasificación
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)


In [2]:
#Carga de datos y lectura

df = pd.read_csv("student_habits_performance.csv")

df = df.drop(columns=['student_id'])

df.head()


,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [3]:
df['aprueba'] = (df['exam_score'] >=60).astype(int)

df['aprueba'].value_counts(normalize=True).round(3)

aprueba
1    0.72
0    0.28
Name: proportion, dtype: float64

In [4]:
df.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score,aprueba
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2,0
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0,1
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3,0
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8,0
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4,1


## Tratamiento de Nulos

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            1000 non-null   int64  
 1   gender                         1000 non-null   str    
 2   study_hours_per_day            1000 non-null   float64
 3   social_media_hours             1000 non-null   float64
 4   netflix_hours                  1000 non-null   float64
 5   part_time_job                  1000 non-null   str    
 6   attendance_percentage          1000 non-null   float64
 7   sleep_hours                    1000 non-null   float64
 8   diet_quality                   1000 non-null   str    
 9   exercise_frequency             1000 non-null   int64  
 10  parental_education_level       909 non-null    str    
 11  internet_quality               1000 non-null   str    
 12  mental_health_rating           1000 non-null   int64  
 13  

In [8]:
moda_educacion = df['parental_education_level'].mode()[0]

df['parental_education_level'] = df['parental_education_level'].fillna(moda_educacion)

df.isnull().sum().sum()

np.int64(0)

## Encoding De variables categóricas

In [ ]:
# OrdinalEncoder para variables con orden real

#diet_quality
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)

#internet_quality
oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)

#parental_education_level
oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)

#--Encoding binario para variables Yes/No
df['part_time_job'] = (df['part_time_job'] =='Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] =='Yes').astype(int)

#--One Hot Encoding para gender (nominal, sin orden)

#Crear una columna binaria por cata categoría de genero
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int)

df.dtypes

age                                int64
study_hours_per_day              float64
social_media_hours               float64
netflix_hours                    float64
part_time_job                      int64
attendance_percentage            float64
sleep_hours                      float64
diet_quality                     float64
exercise_frequency                 int64
parental_education_level         float64
internet_quality                 float64
mental_health_rating               int64
extracurricular_participation      int64
exam_score                       float64
aprueba                            int64
gender_Male                        int64
gender_Other                       int64
dtype: object

# Preparar X e y para clasificación

In [36]:
features_reg = ['study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage',
                'sleep_hours', 'diet_quality', 'exercise_frequency', 'parental_education_level', 'internet_quality',
                'mental_health_rating', 'extracurricular_participation', 'gender_Male', 'gender_Other']

In [37]:
X_clf = df[features_reg]
y_clf = df['aprueba'] #El target binario: 1= aprueba y 0 reprueba

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

scaler_c = StandardScaler()
X_train_c_sc = scaler_c.fit_transform(X_train_c)
X_test_c_sc = scaler_c.transform(X_test_c)

## Entrenamiento del modelo logístico

In [38]:
modelo_log = LogisticRegression(class_weight="balanced", random_state=42)

modelo_log.fit(X_train_c_sc, y_train_c)

#Aplcia un umbral donde 0.5 si aprueba si no = 0
y_pred_log = modelo_log.predict(X_test_c_sc)

y_prob_log = modelo_log.predict_proba(X_test_c_sc)

pd.DataFrame({
    'Real (aprueba)':   y_test_c.values[:10],
    'Predicho':         y_pred_log[:10],
    'P(aprueba=1)':      y_prob_log[:10, 1].round(3)
})

,Real (aprueba),Predicho,P(aprueba=1)
0,0,0,0.000
1,1,1,0.900
2,1,1,0.677
3,1,1,0.972
4,1,0,0.391
5,1,1,0.941
6,1,1,0.998
7,0,0,0.003
8,0,0,0.235
9,0,0,0.336


## Metricas de Evaluación para clasificación

In [39]:
#Accuracy (VP+VN)/Total
#Precision (VP / (VP+FP))
#Recall (VP 7(VP+FN))
#F1-score = Balance entre Precision y Recall

#Glosario
#VP
#VN
#FP = Falso positivio = predije aprueba pero reprueba
#FN

acc = accuracy_score(y_test_c, y_pred_log)
print(f'Accuracy general del modelo: {acc:.1%}')

Accuracy general del modelo: 91.5%


In [40]:
print(classification_report(y_test_c, y_pred_log,
                            target_names=['Reprueba (0)', 'Aprueba (1)']))

              precision    recall  f1-score   support

Reprueba (0)       0.79      0.95      0.86        56
 Aprueba (1)       0.98      0.90      0.94       144

    accuracy                           0.92       200
   macro avg       0.88      0.92      0.90       200
weighted avg       0.93      0.92      0.92       200



## Regresión Lineal

In [ ]:
#División del dataset en entrenamiento y prueba

X_reg = df[features_reg]

y_reg = df['exam_score']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42,
)

scaler_r = StandardScaler()
X_train_r_sc = scaler_r.fit_transform(X_train_r)
X_test_r_sc = scaler_r.transform(X_test_r)

print(f'Entrenamiento: {X_train_r_sc.shape[0]} estudiantes | prueba: {X_test_r_sc.shape[0]} estudiantes')

Entrenamiento: 800 estudiantes | prueba: 200 estudiantes


# Entrenamiento del modelo de Regresión Lineal

In [42]:
modelo_lr = LinearRegression()

modelo_lr.fit(X_train_r_sc, y_train_r)

y_pred_lr = modelo_lr.predict(X_test_r_sc)

pd.DataFrame({
    'Puntaje real':  y_test_r.values[:10],
    'Puntaje Predicho': y_pred_lr[:10].round(1)
})

,Puntaje real,Puntaje Predicho
0,64.2,65.9
1,72.7,74.7
2,79.0,78.5
3,79.5,73.5
4,58.2,61.2
5,53.4,54.8
6,70.8,75.4
7,62.5,55.3
8,36.8,40.9
9,67.6,72.6


## Métricas de Evaluación